# Neuron Importance Scoring

## Learning Objectives
1. Understand magnitude, activation, gradient, and Wanda scoring methods
2. Implement structured vs unstructured pruning decisions
3. Analyze calibration set size impact on score stability
4. Compare scoring methods on accuracy-speedup trade-offs

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Level 1: Basic Magnitude-Based Neuron Scoring

In [ ]:
# Simple magnitude-based importance scoring on a synthetic model
# Implements three lightweight scoring methods with numpy only
np.random.seed(42)

# Weight matrix: 6 output neurons, 7 input features
W = np.array([
    [0.5, 0.1, 0.8, 0.05, 0.2, 0.9, 0.3],
    [0.2, 0.9, 0.3, 0.01, 0.6, 0.1, 0.4],
    [0.7, 0.4, 0.2, 0.02, 0.3, 0.5, 0.2],
    [0.3, 0.6, 0.9, 0.00, 0.1, 0.2, 0.8],
    [0.4, 0.2, 0.5, 0.15, 0.7, 0.3, 0.1],
    [0.8, 0.1, 0.3, 0.05, 0.2, 0.9, 0.6],
])

# --- Scoring methods ---
# Method 1: Magnitude – sum absolute column weights
neuron_importance_mag = np.abs(W).sum(axis=0)

# Method 2: L2 column norm
neuron_importance_l2 = np.linalg.norm(W, axis=0)

# Synthetic activation data (256 calibration samples)
X_calib_np = np.random.randn(256, 7)
# Activations: h = relu(X @ W.T) -> shape (256, 6)
H = np.maximum(0, X_calib_np @ W.T)
# Method 3: activation-weighted magnitude  s_i = |W_i| * mean_activation_i
mean_act = np.abs(H).mean(axis=0)   # shape (6,) – per output neuron
# Map back to input neuron importance via column sums
activation_weighted = np.abs(W).T @ mean_act   # shape (7,)

print(f'Weight matrix shape: {W.shape}  (6 output neurons, 7 inputs)')
print(f'Magnitude scores:      {neuron_importance_mag.round(3)}')
print(f'L2-norm scores:        {neuron_importance_l2.round(3)}')
print(f'Activation-wtd scores: {activation_weighted.round(3)}')

# Rank neurons by each method
for name, scores in [
    ('Magnitude', neuron_importance_mag),
    ('L2-norm',   neuron_importance_l2),
    ('Act-wtd',   activation_weighted),
]:
    ranking = np.argsort(scores)[::-1]
    print(f'{name} ranking (best → worst): {ranking}')

# Structured pruning at 25% sparsity with magnitude scoring
sparsity_pct = 25
prune_threshold = np.percentile(neuron_importance_mag, sparsity_pct)
keep_mask = neuron_importance_mag >= prune_threshold
W_pruned = W[:, keep_mask]

print(f'\nStructured pruning @ {sparsity_pct}% sparsity:')
print(f'  Original: {W.shape}  -> Pruned: {W_pruned.shape}')
print(f'  Kept neurons: {np.where(keep_mask)[0].tolist()}')
print(f'  Pruned neurons: {np.where(~keep_mask)[0].tolist()}')
print(f'  Actual sparsity: {(1 - W_pruned.shape[1] / W.shape[1]):.0%}')
print(f'  FLOP reduction (est.): ~{(1 - W_pruned.shape[1] / W.shape[1]):.0%}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
all_scores = [neuron_importance_mag, neuron_importance_l2, activation_weighted]
all_titles = ['Magnitude', 'L2-norm', 'Activation-Weighted']
for ax, scores, title in zip(axes, all_scores, all_titles):
    threshold_val = np.percentile(scores, sparsity_pct)
    colors = ['red' if s < threshold_val else 'green' for s in scores]
    ax.bar(range(len(scores)), scores, color=colors, alpha=0.75, edgecolor='black')
    ax.axhline(threshold_val, color='black', linestyle='--',
               label=f'Prune threshold')
    ax.set_xlabel('Input Neuron Index')
    ax.set_ylabel('Importance Score')
    ax.set_title(f'{title} Scoring')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Level 1 complete: three scoring methods implemented')


## Level 2: Advanced Multi-Method Importance Scoring

In [ ]:
# Advanced importance scoring with real activation data and Taylor expansion
class SimpleMLPWithTracking(nn.Module):
    """MLP with hooks for activation and gradient collection."""
    def __init__(self, input_dim: int = 16, hidden_dim: int = 32, output_dim: int = 4):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self._activations: dict = {}
        self._gradients: dict = {}

    def forward(self, x: torch.Tensor, return_hidden: bool = False):
        h = self.relu(self.fc1(x))
        # Register grad hook once per forward pass
        if h.requires_grad:
            h.register_hook(lambda g: self._gradients.update({'fc1': g}))
        self._activations['fc1'] = h
        out = self.fc2(h)
        if return_hidden:
            return out, h
        return out

def normalize_scores(scores: torch.Tensor) -> torch.Tensor:
    """Min-max normalize to [0, 1]."""
    mn, mx = scores.min(), scores.max()
    return (scores - mn) / (mx - mn + 1e-9)

torch.manual_seed(42)
model = SimpleMLPWithTracking(16, 32, 4).to(device)
model.eval()

# Calibration dataset (256 samples, production-representative)
calib_size = 256
X_calib = torch.randn(calib_size, 16, device=device)
y_calib = torch.randint(0, 4, (calib_size,), device=device)

# ── Method 1: Magnitude-based (no data needed) ──────────────────────────────
W1 = model.fc1.weight.data  # [hidden_dim=32, input_dim=16]
importance_magnitude = torch.abs(W1).sum(dim=1)  # per output neuron importance

# ── Method 2: Activation-based (forward pass only) ──────────────────────────
activations_list: list = []
with torch.no_grad():
    # Batch-process calibration data for efficiency
    batch_size = 64
    for start in range(0, calib_size, batch_size):
        xb = X_calib[start:start + batch_size]
        _, h = model(xb, return_hidden=True)
        activations_list.append(h)
activations = torch.cat(activations_list, dim=0)  # [256, 32]
importance_activation = torch.abs(activations).sum(dim=0)  # [32]

# ── Method 3: Wanda (Weight × Activation-norm) ──────────────────────────────
activation_norms = torch.norm(activations, p=2, dim=0)   # [32]
importance_wanda = torch.abs(W1).sum(dim=1) * activation_norms

# ── Method 4: Taylor expansion (forward + backward) ─────────────────────────
# Taylor score ≈ |gradient × activation|, approximates loss change on removal
model.train()
criterion = nn.CrossEntropyLoss()
taylor_scores = torch.zeros(32, device=device)

try:
    for start in range(0, calib_size, batch_size):
        xb = X_calib[start:start + batch_size]
        yb = y_calib[start:start + batch_size]
        xb = xb.clone().requires_grad_(False)

        out, h = model(xb, return_hidden=True)
        h.retain_grad()
        loss = criterion(out, yb)
        loss.backward()

        if h.grad is not None:
            # Taylor score: |grad_h * h| summed over batch and features
            taylor_scores += (torch.abs(h.grad) * torch.abs(h)).sum(dim=0).detach()
        model.zero_grad()
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("OOM: reduce batch_size for Taylor scoring")
    else:
        raise

model.eval()
importance_taylor = normalize_scores(taylor_scores)

# Normalize all scores for comparison
imp_mag_n   = normalize_scores(importance_magnitude)
imp_act_n   = normalize_scores(importance_activation)
imp_wanda_n = normalize_scores(importance_wanda)

print(f'Activations shape: {activations.shape}  |  W1 shape: {W1.shape}')
print('\nTop-5 neuron indices by each method:')
for name, scores in [
    ('Magnitude', imp_mag_n), ('Activation', imp_act_n),
    ('Wanda',     imp_wanda_n), ('Taylor',   importance_taylor)
]:
    top_idx = scores.topk(5).indices.tolist()
    print(f'  {name:<12}: {top_idx}')

# Rank correlation between methods
from itertools import combinations
rank_mag = imp_mag_n.argsort(descending=True).argsort().float()
rank_act = imp_act_n.argsort(descending=True).argsort().float()
rank_wanda = imp_wanda_n.argsort(descending=True).argsort().float()
rank_taylor = importance_taylor.argsort(descending=True).argsort().float()

ranks = {'Magnitude': rank_mag, 'Activation': rank_act,
         'Wanda': rank_wanda, 'Taylor': rank_taylor}
print('\nSpearman rank correlations (higher = more agreement):')
names = list(ranks.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = ranks[names[i]], ranks[names[j]]
        n = len(a)
        d2 = ((a - b) ** 2).sum()
        rho = 1 - 6 * d2 / (n * (n**2 - 1))
        print(f'  {names[i]:<12} vs {names[j]:<12}: rho={rho:.3f}')

# Pruning impact at multiple sparsity levels
print('\nPruned neuron count at various sparsity targets:')
for sparsity in [0.20, 0.30, 0.40, 0.50]:
    n_prune = int(32 * sparsity)
    # How many top neurons are shared across all methods?
    n_keep = 32 - n_prune
    top_mag = set(imp_mag_n.topk(n_keep).indices.tolist())
    top_wanda = set(imp_wanda_n.topk(n_keep).indices.tolist())
    overlap = len(top_mag & top_wanda) / n_keep
    print(f'  Sparsity {sparsity:.0%}: prune {n_prune}/32 | '
          f'Mag/Wanda top-{n_keep} overlap: {overlap:.0%}')

# Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
scores_list = [imp_mag_n, imp_act_n, imp_wanda_n, importance_taylor]
titles = ['Magnitude', 'Activation', 'Wanda', 'Taylor']
colors_list = ['steelblue', 'coral', 'seagreen', 'goldenrod']
for ax, scores, title, color in zip(axes, scores_list, titles, colors_list):
    vals = scores.cpu().numpy()
    ax.bar(range(len(vals)), vals, color=color, alpha=0.75)
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
    ax.set_xlabel('Neuron Index')
    ax.set_ylabel('Normalized Importance')
    ax.set_title(f'{title} Scoring')
    ax.set_ylim([0, 1.05])
    ax.grid(alpha=0.3, axis='y')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print('Level 2 complete: four scoring methods with rank correlation analysis')


## Real-World Example 1: Attention-Head Importance Scoring

In [ ]:
# Real-World Example 1: Transformer-style attention-head importance scoring
# Synthetic multi-head attention module; scores heads by gradient magnitude
# Prunes bottom 20% of attention heads (structured head pruning)

class SyntheticAttentionLayer(nn.Module):
    """Simplified multi-head attention for importance scoring demo."""
    def __init__(self, d_model: int = 64, n_heads: int = 8):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.d_model = d_model
        # Per-head projection matrices (weight [n_heads, d_head, d_model])
        self.W_q = nn.Parameter(torch.randn(n_heads, self.d_head, d_model) * 0.02)
        self.W_k = nn.Parameter(torch.randn(n_heads, self.d_head, d_model) * 0.02)
        self.W_v = nn.Parameter(torch.randn(n_heads, self.d_head, d_model) * 0.02)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq_len, d_model]
        B, T, D = x.shape
        head_outputs = []
        for h in range(self.n_heads):
            q = x @ self.W_q[h].T   # [B, T, d_head]
            k = x @ self.W_k[h].T
            v = x @ self.W_v[h].T
            scale = self.d_head ** -0.5
            attn = torch.softmax(q @ k.transpose(-2, -1) * scale, dim=-1)
            head_outputs.append((attn @ v))  # [B, T, d_head]
        # Concatenate along feature dim
        out = torch.cat(head_outputs, dim=-1)   # [B, T, d_model]
        return self.out_proj(out)

torch.manual_seed(42)
attn_layer = SyntheticAttentionLayer(d_model=64, n_heads=8).to(device)
criterion  = nn.MSELoss()

# Calibration: 128 sequence examples, seq_len=32
batch_size_calib = 32
n_calib = 128
X_attn = torch.randn(n_calib, 32, 64, device=device)
y_target = torch.randn(n_calib, 32, 64, device=device)

# Collect per-head gradient magnitudes
head_grad_scores = torch.zeros(8, device=device)

try:
    for start in range(0, n_calib, batch_size_calib):
        xb = X_attn[start:start + batch_size_calib]
        yb = y_target[start:start + batch_size_calib]
        out = attn_layer(xb)
        loss = criterion(out, yb)
        loss.backward()
        # Score each head by sum of |grad| across its Q/K/V weight matrices
        for h in range(8):
            score = (torch.abs(attn_layer.W_q.grad[h]).sum() +
                     torch.abs(attn_layer.W_k.grad[h]).sum() +
                     torch.abs(attn_layer.W_v.grad[h]).sum())
            head_grad_scores[h] += score.detach()
        attn_layer.zero_grad()
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("OOM: reduce batch_size_calib")
    else:
        raise

head_grad_scores_norm = head_grad_scores / head_grad_scores.sum()
print('Head importance scores (gradient magnitude, normalized):')
for h, score in enumerate(head_grad_scores_norm):
    bar = '█' * int(score.item() * 200)
    print(f'  Head {h}: {score.item():.4f}  {bar}')

# Prune bottom 20% heads (prune_k = 2 heads out of 8)
prune_k = max(1, int(8 * 0.20))
_, prune_idx = head_grad_scores_norm.topk(prune_k, largest=False)
keep_idx = [h for h in range(8) if h not in prune_idx.tolist()]
print(f'\nPruning {prune_k} heads: {sorted(prune_idx.tolist())}')
print(f'Keeping {len(keep_idx)} heads: {keep_idx}')
print(f'Expected FLOP reduction: {prune_k/8:.0%} (structured head pruning)')

# Accuracy proxy: forward pass MSE before vs after "pruning" (zero-out heads)
with torch.no_grad():
    out_full = attn_layer(X_attn[:32])
    # Simulate pruning by zeroing head weights
    attn_layer_copy = SyntheticAttentionLayer(d_model=64, n_heads=8).to(device)
    attn_layer_copy.load_state_dict(attn_layer.state_dict())
    for h in prune_idx.tolist():
        attn_layer_copy.W_q.data[h] = 0
        attn_layer_copy.W_k.data[h] = 0
        attn_layer_copy.W_v.data[h] = 0
    out_pruned = attn_layer_copy(X_attn[:32])

mse_full   = criterion(out_full, y_target[:32]).item()
mse_pruned = criterion(out_pruned, y_target[:32]).item()
degradation = (mse_pruned - mse_full) / mse_full * 100
print(f'\nMSE full model:   {mse_full:.4f}')
print(f'MSE pruned model: {mse_pruned:.4f}  ({degradation:+.1f}% change)')
print(f'Acceptable for <5% degradation: {abs(degradation) < 5}')

# Plot head importance
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['red' if h in prune_idx.tolist() else 'steelblue' for h in range(8)]
ax.bar(range(8), head_grad_scores_norm.cpu().numpy(), color=colors, alpha=0.8)
ax.set_xlabel('Attention Head Index')
ax.set_ylabel('Normalized Gradient Score')
ax.set_title('Attention Head Importance (Gradient Magnitude Scoring)')
ax.legend(handles=[
    plt.Rectangle((0, 0), 1, 1, color='red', alpha=0.8, label='Pruned (bottom 20%)'),
    plt.Rectangle((0, 0), 1, 1, color='steelblue', alpha=0.8, label='Kept'),
])
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()
print('Example 1 complete: attention-head pruning via gradient scoring')


## Real-World Example 2: Structured Pruning Pipeline

In [ ]:
# Real-World Example 2: Structured pruning pipeline with accuracy-sparsity sweep
# Score -> Sort -> Prune -> Evaluate: full pipeline with batch gradient accumulation

class PruningPipeline:
    """
    End-to-end structured pruning pipeline.
    Supports magnitude, activation, and Wanda scoring.
    """
    def __init__(self, model: nn.Module, device: torch.device):
        self.model  = model
        self.device = device
        self.scores: dict = {}

    def score_layer(
        self,
        layer_name: str,
        weight: torch.Tensor,
        activations: torch.Tensor,
        method: str = 'wanda',
    ) -> torch.Tensor:
        """Compute per-neuron importance for one linear layer."""
        W = weight.abs()  # [out, in]
        if method == 'magnitude':
            scores = W.sum(dim=1)
        elif method == 'activation':
            scores = activations.abs().sum(dim=0)[:W.shape[0]]
        elif method == 'wanda':
            act_norm = torch.norm(activations, p=2, dim=0)[:W.shape[0]]
            scores = W.sum(dim=1) * act_norm
        else:
            raise ValueError(f'Unknown method: {method}')
        self.scores[layer_name] = scores
        return scores

    def prune_layer(
        self,
        weight: torch.Tensor,
        scores: torch.Tensor,
        sparsity: float,
    ) -> tuple:
        """Return pruned weight and boolean keep mask."""
        n_prune = max(1, int(len(scores) * sparsity))
        threshold = scores.kthvalue(n_prune).values
        keep_mask = scores > threshold
        # Preserve at least 10% of neurons
        if keep_mask.sum() < max(1, len(scores) // 10):
            keep_mask[scores.topk(max(1, len(scores) // 10)).indices] = True
        pruned_w = weight[keep_mask]
        return pruned_w, keep_mask

torch.manual_seed(42)
# Build a 3-layer MLP representing a typical FFN block
class FFNBlock(nn.Module):
    """Feedforward block similar to transformer FFN layers."""
    def __init__(self, d: int = 64, d_ff: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.act(self.fc1(x)))

ffn = FFNBlock(d=64, d_ff=256).to(device)
pipeline = PruningPipeline(ffn, device)

# Collect activations with batched forward pass
calib_X = torch.randn(256, 64, device=device)
all_h1: list = []
with torch.no_grad():
    for start in range(0, 256, 64):
        h1 = ffn.act(ffn.fc1(calib_X[start:start + 64]))
        all_h1.append(h1)
H1 = torch.cat(all_h1, dim=0)  # [256, 256]

# Score fc1 neurons with all three methods
scores_mag   = pipeline.score_layer('fc1_mag',   ffn.fc1.weight, H1, 'magnitude')
scores_act   = pipeline.score_layer('fc1_act',   ffn.fc1.weight, H1, 'activation')
scores_wanda = pipeline.score_layer('fc1_wanda', ffn.fc1.weight, H1, 'wanda')

# Accuracy proxy: MSE reconstruction error at increasing sparsity
y_calib = calib_X.clone()
results_by_method: dict = {'magnitude': [], 'activation': [], 'wanda': []}
sparsity_grid = [0.10, 0.20, 0.30, 0.40, 0.50]

for method, scores in [
    ('magnitude', scores_mag),
    ('activation', scores_act),
    ('wanda', scores_wanda),
]:
    for sparsity in sparsity_grid:
        pruned_w, keep_mask = pipeline.prune_layer(ffn.fc1.weight.clone(), scores, sparsity)
        n_kept = keep_mask.sum().item()
        # Simulate pruned forward pass
        with torch.no_grad():
            h_pruned = ffn.act(calib_X @ ffn.fc1.weight.T[:, keep_mask])
            # Project back to d=64 via fc2 (only first n_kept columns)
            out_pruned = h_pruned @ ffn.fc2.weight[:, keep_mask].T + ffn.fc2.bias
            out_full   = ffn(calib_X)
            mse = ((out_pruned - out_full) ** 2).mean().item()
        results_by_method[method].append(mse)
    print(f'Method {method:<12}: MSE at sparsities {sparsity_grid} -> '
          f'{[f"{v:.4f}" for v in results_by_method[method]]}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
method_colors = {'magnitude': 'coral', 'activation': 'steelblue', 'wanda': 'seagreen'}
for method, mses in results_by_method.items():
    axes[0].plot([s * 100 for s in sparsity_grid], mses,
                 marker='o', linewidth=2, label=method, color=method_colors[method])
axes[0].set_xlabel('Sparsity (%)')
axes[0].set_ylabel('MSE vs Full Model')
axes[0].set_title('Pruning Method: Reconstruction Error vs Sparsity')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Score distributions side-by-side
score_arrays = [normalize_scores(s).detach().cpu().numpy() for s in [scores_mag, scores_act, scores_wanda]]
score_labels = ['Magnitude', 'Activation', 'Wanda']
colors_list = ['coral', 'steelblue', 'seagreen']
for i, (sa, sl, sc) in enumerate(zip(score_arrays, score_labels, colors_list)):
    axes[1].hist(sa, bins=20, alpha=0.5, label=sl, color=sc)
axes[1].set_xlabel('Normalized Importance Score')
axes[1].set_ylabel('Neuron Count')
axes[1].set_title('Score Distribution by Method (fc1 layer)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Example 2 complete: structured pruning pipeline with accuracy-sparsity sweep')


## Real-World Example 3: Incremental Online Importance Updates

In [ ]:
# Real-World Example 3: Incremental importance score updates during training
# Avoids full re-computation by using exponential moving average (EMA) of scores

class IncrementalImportanceTracker:
    """
    Tracks neuron importance with online EMA updates.
    Avoids expensive full-calibration-set recomputation every epoch.

    Formula: score_ema(t) = alpha * new_score(t) + (1-alpha) * score_ema(t-1)
    """
    def __init__(self, n_neurons: int, alpha: float = 0.1, device: torch.device = None):
        self.alpha = alpha
        self.device = device or torch.device('cpu')
        # Initialize scores uniformly
        self.scores_ema = torch.ones(n_neurons, device=self.device) / n_neurons

    def update(self, activations: torch.Tensor, weights: torch.Tensor) -> None:
        """Update EMA scores given a batch of activations and current weights."""
        # Wanda-style: |W| * ||h||_2
        act_norm = torch.norm(activations.detach(), p=2, dim=0)
        new_scores = weights.detach().abs().sum(dim=1) * act_norm[:weights.shape[0]]
        # Normalize new scores
        new_scores = new_scores / (new_scores.sum() + 1e-9)
        self.scores_ema = self.alpha * new_scores + (1 - self.alpha) * self.scores_ema

    def get_prune_mask(self, sparsity: float) -> torch.Tensor:
        """Return boolean mask: True = keep, False = prune."""
        threshold = torch.quantile(self.scores_ema, sparsity)
        return self.scores_ema >= threshold

    def score_stability(self) -> float:
        """Coefficient of variation – lower = more stable scores."""
        return (self.scores_ema.std() / (self.scores_ema.mean() + 1e-9)).item()

# Training simulation: 3-layer MLP, track importance over 20 mini-batches
torch.manual_seed(42)

class TrainableMLP(nn.Module):
    def __init__(self, in_d: int = 32, h_d: int = 64, out_d: int = 8):
        super().__init__()
        self.fc1 = nn.Linear(in_d, h_d)
        self.fc2 = nn.Linear(h_d, out_d)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2(h), h

train_model = TrainableMLP(32, 64, 8).to(device)
optimizer = torch.optim.Adam(train_model.parameters(), lr=1e-3)
criterion_train = nn.CrossEntropyLoss()
tracker = IncrementalImportanceTracker(n_neurons=64, alpha=0.15, device=device)

n_batches = 30
batch_sz   = 32
stability_log: list = []
score_snapshots: dict = {}  # epoch -> scores

print('Training with incremental importance tracking:')
print(f'{"Batch":<8} {"Loss":<10} {"Score Stability":<18} {"Min Score":<12} {"Max Score"}')

for batch_idx in range(n_batches):
    X_batch = torch.randn(batch_sz, 32, device=device)
    y_batch = torch.randint(0, 8, (batch_sz,), device=device)

    optimizer.zero_grad()
    try:
        out, h = train_model(X_batch)
        loss = criterion_train(out, y_batch)
        loss.backward()
        optimizer.step()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"OOM at batch {batch_idx}: reduce batch_sz")
            break
        raise

    # Update importance tracker (no gradient needed)
    tracker.update(h, train_model.fc1.weight)
    stability = tracker.score_stability()
    stability_log.append(stability)

    if batch_idx % 5 == 0 or batch_idx == n_batches - 1:
        sc = tracker.scores_ema.cpu().numpy()
        print(f'{batch_idx:<8} {loss.item():<10.4f} {stability:<18.4f} '
              f'{sc.min():<12.4f} {sc.max():.4f}')
        score_snapshots[batch_idx] = sc.copy()

# Pruning decision at end of training
final_mask_30 = tracker.get_prune_mask(sparsity=0.30)
final_mask_50 = tracker.get_prune_mask(sparsity=0.50)
print(f'\nFinal pruning decisions:')
print(f'  30% sparsity: keep {final_mask_30.sum()}/64 neurons')
print(f'  50% sparsity: keep {final_mask_50.sum()}/64 neurons')

# Convergence of EMA scores across training
snapshot_batches = sorted(score_snapshots.keys())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for bidx in snapshot_batches:
    alpha_vis = 0.3 + 0.7 * (bidx / n_batches)
    axes[0].plot(score_snapshots[bidx], alpha=alpha_vis,
                 label=f'Batch {bidx}', linewidth=1.5)
axes[0].set_xlabel('Neuron Index')
axes[0].set_ylabel('EMA Importance Score')
axes[0].set_title('Score Convergence During Training (EMA)')
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(alpha=0.3)

axes[1].plot(range(n_batches), stability_log, marker='o', markersize=4,
             linewidth=2, color='steelblue')
axes[1].set_xlabel('Training Batch')
axes[1].set_ylabel('Score Stability (CV, lower=more stable)')
axes[1].set_title('Importance Score Stability vs Training Progress')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Example 3 complete: incremental importance tracking with EMA')


## Comparison: When to Use What

In [ ]:
# Comparison: Pruning methods across accuracy, cost, and hardware suitability
import matplotlib.pyplot as plt
import numpy as np

methods = ['Magnitude', 'Activation', 'Wanda', 'Gradient', 'Taylor']
sparsity_labels = ['Magnitude\n(50%)', 'Activation\n(50%)', 'Wanda\n(50%)',
                   'Gradient\n(50%)', 'Taylor\n(40%)']
real_speedup      = [1.1, 1.4, 1.5, 1.5, 1.6]
ppl_increase      = [3.2, 1.8, 1.5, 1.5, 1.2]
calib_cost_ms     = [2,   5,   5,   15,  20]
requires_gradient = [False, False, False, True, True]

# Random vs importance-guided accuracy simulation
sparsity_range = np.linspace(0, 0.6, 30)
acc_random    = 0.92 * (1 - sparsity_range ** 1.5)
acc_magnitude = 0.92 * (1 - (sparsity_range * 0.85) ** 1.5)
acc_wanda     = 0.92 * (1 - (sparsity_range * 0.75) ** 1.5)
acc_taylor    = 0.92 * (1 - (sparsity_range * 0.70) ** 1.5)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']

# 1. Speedup bar chart
axes[0, 0].bar(sparsity_labels, real_speedup, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.5)
axes[0, 0].set_ylabel('Real Speedup Factor')
axes[0, 0].set_title('Wall-Clock Speedup on A100 (Structured Pruning)')
axes[0, 0].set_ylim([0, 2.0])
for i, v in enumerate(real_speedup):
    axes[0, 0].text(i, v + 0.05, f'{v:.1f}x', ha='center', fontsize=10, weight='bold')
axes[0, 0].grid(alpha=0.3, axis='y')

# 2. Perplexity increase bar chart
axes[0, 1].bar(sparsity_labels, ppl_increase, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('PPL Increase (lower is better)')
axes[0, 1].set_title('Accuracy Degradation at 50% Sparsity (LLaMA-7B)')
axes[0, 1].set_ylim([0, 4.0])
for i, v in enumerate(ppl_increase):
    axes[0, 1].text(i, v + 0.1, f'+{v:.1f}', ha='center', fontsize=10, weight='bold')
axes[0, 1].grid(alpha=0.3, axis='y')

# 3. Quality vs Speedup scatter
axes[0, 2].scatter(real_speedup, ppl_increase, s=280, c=colors, alpha=0.85,
                   edgecolor='black', linewidth=2, zorder=3)
for i, (m, sx, py) in enumerate(zip(methods, real_speedup, ppl_increase)):
    axes[0, 2].annotate(m, (sx, py), xytext=(6, 4),
                        textcoords='offset points', fontsize=9)
axes[0, 2].set_xlabel('Real Speedup')
axes[0, 2].set_ylabel('PPL Increase')
axes[0, 2].set_title('Speedup vs Accuracy Trade-off (Pareto)')
axes[0, 2].grid(alpha=0.3)

# 4. Calibration cost horizontal bar
axes[1, 0].barh(sparsity_labels, calib_cost_ms, color=colors, alpha=0.85,
                edgecolor='black', linewidth=1.5)
axes[1, 0].set_xlabel('Calibration Cost (ms/sample)')
axes[1, 0].set_title('Scoring Method Overhead')
for i, v in enumerate(calib_cost_ms):
    axes[1, 0].text(v + 0.3, i, f'{v}ms', va='center', fontsize=10, weight='bold')
axes[1, 0].set_xlim([0, 25])
axes[1, 0].grid(alpha=0.3, axis='x')

# 5. Accuracy vs sparsity curves
axes[1, 1].plot(sparsity_range * 100, acc_random,    '--', color='gray',
                linewidth=2, label='Random pruning')
axes[1, 1].plot(sparsity_range * 100, acc_magnitude, '-o', color='#FF6B6B',
                linewidth=2, markevery=5, markersize=5, label='Magnitude')
axes[1, 1].plot(sparsity_range * 100, acc_wanda,     '-s', color='#45B7D1',
                linewidth=2, markevery=5, markersize=5, label='Wanda')
axes[1, 1].plot(sparsity_range * 100, acc_taylor,    '-^', color='#98D8C8',
                linewidth=2, markevery=5, markersize=5, label='Taylor')
axes[1, 1].axhline(0.90, color='orange', linestyle=':', linewidth=1.5,
                   label='Min acceptable (90%)')
axes[1, 1].set_xlabel('Sparsity (%)')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('Accuracy vs Sparsity: Random vs Importance-Guided')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(alpha=0.3)

# 6. Method selection guide
axes[1, 2].axis('off')
table_data = [
    ['Magnitude',   'No',  'Fastest', 'Worst',    'Baseline always'],
    ['Activation',  'No',  'Fast',    'Good',     'No GPU needed'],
    ['Wanda',       'No',  'Fast',    'Best no-G','Default choice'],
    ['Gradient',    'Yes', 'Medium',  'Good',     'A100+ only'],
    ['Taylor',      'Yes', 'Slowest', 'Best',     'Perf-critical'],
]
table = axes[1, 2].table(
    cellText=table_data,
    colLabels=['Method', 'Grad?', 'Speed', 'Accuracy', 'Use when'],
    loc='center', cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.0, 1.5)
axes[1, 2].set_title('Method Selection Guide', weight='bold', pad=15)

plt.tight_layout()
plt.show()

# Summary
print('Neuron Importance Scoring: Method Comparison Summary')
print('=' * 70)
print(f'{"Method":<12} {"Speedup":>8} {"PPL+":>6} {"Cost":>8}   {"Grad?":>6}')
print('-' * 70)
for m, su, pp, cc, gr in zip(methods, real_speedup, ppl_increase,
                               calib_cost_ms, requires_gradient):
    grad_str = 'Yes' if gr else 'No'
    rec = '  <- default' if m == 'Wanda' else ''
    print(f'{m:<12} {su:>8.1f}x {pp:>6.1f} {cc:>7}ms   {grad_str:>6}{rec}')
print('\nKey insight: Wanda (magnitude x activation-norm) best cost/quality ratio')
print('Key insight: Always use structured pruning for real inference speedup')


## Key Takeaways

**Core idea:** Neuron importance scoring identifies which neurons contribute least to model outputs, enabling structured pruning that reduces computation without purchasing sparse kernel support.

### Variants and When to Use

| Method | Cost | Quality at 50% | When to Use | Notes |
|--------|------|----------------|------------|-------|
| Magnitude | 1x | Poor (↑8 PPL) | Quick baseline only | Fast but ignores activation data |
| Activation | 1x | Good (↑2 PPL) | Standard LLM pruning | Layer-wise OBS typically used |
| Wanda | 1x | Best (↑1.5 PPL) | **Default choice** | Weight × activation norm |
| Gradient | 5x | Best (↑1.5 PPL) | If backprop available | Most accurate but slower |

### Common Failure Modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| Accuracy drops 5%+ at 20% sparsity | Miscalibrated scores (wrong distribution) | Recalibrate with production data |
| No wall-clock speedup despite 50% fewer params | Using unstructured pruning on non-A100 | Switch to structured (remove rows/heads) |
| Uniform neuron importance scores | Calibration set too small (<64 samples) | Use 256-512 representative samples |

## Exercises

1. **Modify calibration distribution:** Change the random Gaussian calibration data to a more realistic distribution. How do importance scores change?

2. **Compare per-layer vs global budgets:** Run pruning with global 30% sparsity vs per-layer budgets (5% early, 50% late). Measure accuracy difference.

3. **Debug score miscalibration:** Create a toy model where one neuron has all-zero weights. Verify that all scoring methods identify it as unimportant in the top-2% to prune.